# Which of these four runs produced the number in the deck?

Four analyses ran last week. Two were the same configuration; one moved a dose level; one
extended the window. Nobody wrote it down, because at the time everyone knew. A month later
a figure is being challenged and the only honest answer available is *probably the third one*.

That is the failure `Spec` exists to make impossible. Every configuration object in axiom
inherits it, and a `Spec` is immutable, rejects unknown fields, serializes through one JSON
envelope, hashes stably across processes, and can diff itself against another of its type.
An analysis is then **the set of content hashes it touched** — a claim that can be checked
rather than remembered.

In [ ]:
import json

from axiom.core import (
    D,
    SchemaVersionError,
    Spec,
    SpecDiff,
    SpecError,
    TimeWindow,
    Treatment,
    UnknownSpecError,
    load_spec,
    spec_type_name,
)

from axiom.display import enable, table

import sys; sys.path[:0] = ["..", "../.."]  # nbs/ is on the path either way
from _style import caption, heat

enable();  # every axiom result renders itself from here on

## Defining one

Subclass `Spec` with pydantic fields. Validation runs once, at construction.

In [ ]:
class ProbeDesign(Spec):
    """A toy spec: which treatment to probe, at which dose levels, over which window."""

    treatment: Treatment
    levels: tuple[float, ...]
    window: TimeWindow
    replicate: int = 1


design = ProbeDesign(
    treatment=Treatment(name="fertilizer", dimension=D.currency, unit="USD"),
    levels=(0.0, 50.0, 100.0),
    window=TimeWindow(start=0, stop=8),
)
design

In [ ]:
try:
    design.replicate = 2
except Exception as e:  # pydantic raises ValidationError on a frozen model
    print(type(e).__name__, "- specs are immutable; use model_copy(update=...)")

design_2 = design.model_copy(update={"replicate": 2})
print(design_2.replicate)

## The envelope

`to_json()` writes `{"spec": "<module>:<Class>", "schema_version": ..., "data": {...}}`.
`from_json` resolves the class by *name*, never by executing stored code, and refuses to
load a different class than the one you asked for. A configuration format that can execute
what it loads is a configuration format that can be a security advisory.

In [ ]:
print(spec_type_name(ProbeDesign))
print(design.to_json(indent=2)[:400], "...")

In [ ]:
back = ProbeDesign.from_json(design.to_json())
print("round-trips equal:", back == design)

generic = load_spec(design.to_json())   # when you do not know the class in advance
print("load_spec gives the concrete type:", type(generic).__name__)

try:
    TimeWindow.from_json(design.to_json())
except UnknownSpecError as e:
    print("wrong class refused:", e)

try:
    Spec.from_json('{"not": "an envelope"}')
except SpecError as e:
    print("bad envelope refused:", e)

## Content hashes

`content_hash()` is blake2b-256 over the canonical envelope (sorted keys, no whitespace,
floats via `repr`, `Fraction` as `"p/q"`). It does not depend on field order, dict insertion
order, `PYTHONHASHSEED`, or the process. Two specs are the same spec iff their hashes match.

In [ ]:
h = design.content_hash()
print(h)
print("stable across copies:", design.model_copy().content_hash() == h)
print("changes with content:", design_2.content_hash() == h)
print("hash() works too, so specs can key dicts and sets:", hash(design) == hash(back))

### The four runs, settled

Now the question at the top of the notebook is one line of code. Below, every pair of last
week's runs, compared by content hash: **1 means the same run**. Run A and run D are
byte-identical configurations that were saved under different filenames; nothing else is.

In [ ]:
runs = {
    "A · baseline": design,
    "B · extra replicate": design_2,
    "C · shifted top dose": design.model_copy(update={"levels": (0.0, 50.0, 150.0)}),
    "D · re-saved baseline": ProbeDesign.from_json(design.to_json()),
}
names = list(runs)
same = [[float(a.content_hash() == b.content_hash()) for b in runs.values()] for a in runs.values()]

fig = heat(
    same, names, names,
    text_fmt="{:.0f}",
    colorbar_title="identical",
    title="Which of these runs is the same run?",
    subtitle="pairwise equality of content hashes — no filenames, no memory, no argument",
    height=320,
)
caption(fig, "A and D are the same configuration under two filenames. B and C each differ "
             "in exactly one field, and the hash does not care which — it only reports that "
             "they are different runs.")

## Diffs

The hash says *whether*; `diff` says *what*. It returns a `SpecDiff` with dotted paths, which
is what the assumption ledger uses to name which facet of an estimand differed.

In [ ]:
d: SpecDiff = design.diff(
    design.model_copy(update={"levels": (0.0, 50.0, 150.0), "window": TimeWindow(start=0, stop=12)})
)
print(d.is_empty)
table(
    [[path, repr(left), repr(right)] for path, (left, right) in d.changed.items()],
    headers=("field", "before", "after"),
)

In [ ]:
fields = sorted({path for run in runs.values() for path in design.diff(run).changed})
changed = [[float(path in design.diff(run).changed) for run in runs.values()] for path in fields]

fig = heat(
    changed, names, fields,
    text_fmt="{:.0f}",
    colorbar_title="differs from A",
    title="…and in exactly which field",
    subtitle="every run against the baseline, by dotted path",
    height=280,
)
caption(fig, "One field per run, named. This is the difference between 'the results moved' "
             "and 'the top dose moved from 100 to 150, and the results moved'.")

## Schema versions and migrations

Each class carries a `SCHEMA_VERSION`. Loading a payload written at an older version either
applies a registered migration or raises `SchemaVersionError` naming both versions. It never
silently coerces — that is how saved analyses rot at the first field rename: the payload
still loads, the renamed field takes its default, and a year of stored runs quietly become a
year of runs with `replicates=1`.

In [ ]:
class ProbeDesignV2(Spec):
    SCHEMA_VERSION = "2"
    treatment: Treatment
    levels: tuple[float, ...]
    window: TimeWindow
    replicates: int = 1          # renamed from `replicate`


# a payload written by the v1 class
v1_payload = {
    "spec": spec_type_name(ProbeDesignV2),
    "schema_version": "1",
    "data": {**design.to_dict()},
}

try:
    ProbeDesignV2.from_json(json.dumps(v1_payload))
except SchemaVersionError as e:
    print("no migration:", e)

ProbeDesignV2.register_migration("1", "2", lambda d: {**{k: v for k, v in d.items() if k != "replicate"}, "replicates": d["replicate"]})
migrated = ProbeDesignV2.from_json(json.dumps(v1_payload))
print("migrated:", migrated.replicates, type(migrated).__name__)

## Discovery

`Spec.subclasses()` lists every spec class the process knows about. The round-trip gate
(`tests/contracts/test_spec_roundtrip.py`) walks this list and fails if any class lacks an
example in its factory table — so a new spec cannot be added without something proving it
survives a save and a load.

In [ ]:
[c.__name__ for c in Spec.subclasses() if c.__module__.startswith("axiom.core")]

## What this bought you

A run is identified by what it *was*, not by where it was saved. Two configurations are the
same or they differ in named fields; an old payload either migrates deliberately or refuses
to load; and nothing in the chain can execute code it read off disk.

`nbs/io/01-save-load-an-analysis.ipynb` is where these hashes become a provenance record,
and `nbs/estimands/02-transfer-plans.ipynb` is where a diff between two specs becomes the
list of assumptions a transfer needs.